<a href="https://colab.research.google.com/github/chetools/CHE4061_Spring2026/blob/main/MESHDistillation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!wget -N -q https://raw.githubusercontent.com/chetools/chetools/main/tools/che5.ipynb -O che5.ipynb
%run che5.ipynb

In [ ]:
import numpy as np
import jax
import jax.numpy as jnp
jax.config.update("jax_enable_x64", True)
from scipy.optimize import root_scalar, minimize_scalar, bracket
from scipy.optimize import root
from scipy.special import expit, logit
from plotly.subplots import make_subplots
from scipy.interpolate import Akima1DInterpolator
np.set_printoptions(precision=5)

In [ ]:
R=8.314
p=Props(['Methanol', 'Ethanol', 'Isopropanol', 'Water'])

In [ ]:
def gamma(x,T):
    tau = p.NRTL_A + p.NRTL_B/T + p.NRTL_C*np.log(T) + p.NRTL_D*T
    G=np.exp(-p.NRTL_alpha*tau)
    xG = x@G
    xtauG_xG = (x@(tau*G))/xG
    return np.exp(xtauG_xG + x@((G*(tau - xtauG_xG[None,:])/xG[None,:]).T))

In [ ]:
def dewP_ideal(y, T):
    P=1./(np.sum(y/p.Pvap(T)))
    return P, y*P/p.Pvap(T)

def dewT_ideal(y, P):

    def P_dev(T):
        return dewP_ideal(y, T)[0] - P
    T = root(P_dev, 300.).x[0]

    return T,  y*P/p.Pvap(T)

In [ ]:
xD = jnp.array([0.9, 0.08, 0.01, 0.01])

In [ ]:
bubbleP_NRTL(xD, 320.)

(Array(46012.90615, dtype=float64),
 Array([0.94686, 0.04472, 0.00402, 0.0044 ], dtype=float64))

In [ ]:
dewP_ideal(xD, T)

(Array(155974.19539, dtype=float64),
 Array([0.82345, 0.12315, 0.01817, 0.03523], dtype=float64))

In [ ]:
def bubbleP_NRTL(x, T):
    Pi= x*gamma(x,T)*p.Pvap(T)
    P=np.sum(Pi)
    return P, Pi/P

def bubbleT_NRTL(x, P):

    def f(T):
        return bubbleP_NRTL(x,T)[0]-P

    #mole-fraction weighted boiling points of each component at P
    #boiling points determined via Clausius Clapeyron, using the Hvap at the normal bp
    #for each component.  p.Hvap returns the heat of vaporization of all components for each
    #temperature if an array of temperatures is given.
    Tguess=np.dot(x,1/(1/p.Tbn-np.log(P/101325)*R/np.diagonal(p.Hvap(p.Tbn))))

    res=root_scalar(f, x0=Tguess, method='secant')
    if not(res.converged):
        return "FAIL", res
    T=root_scalar(f, x0=Tguess, method='secant').root
    Pi= x*gamma(x,T)*p.Pvap(T)
    P=np.sum(Pi)

    return T, Pi/P



In [ ]:
def dewP_NRTL(y, T):
    def f(vec):
        P = vec[0]
        x = expit(vec[1:])  #Ensures that mole fractions are between 0 and 1

        fug_eqs = x*gamma(x,T) * p.Pvap(T)  - y*P
        xsum_eq = 1. - np.sum(x)

        return np.r_[fug_eqs, xsum_eq]

    #Assume ideal liquid dewP calculation for initial guess of P and liquid phase composition
    Pguess, xguess= dewP_ideal(y,T)

    #f (function to zero) maps values from -inf to inf, to values between 0 and 1
    #so xguess is mapped via logit which is the inverse function of expit
    v0 = np.r_[Pguess, logit(xguess)]
    res=root(f, v0)
    if not(res.success):
        return "FAILURE", res
    return res.x[0], expit(res.x[1:])

In [ ]:
def dewT_NRTL(y, P):
    def f(vec):
        T = vec[0]
        x = expit(vec[1:])  #Ensures that mole fractions are between 0 and 1

        fug_eqs = x*gamma(x,T) * p.Pvap(T)  - y*P
        xsum_eq = 1. - np.sum(x)

        return np.r_[fug_eqs, xsum_eq]

    #Assume ideal liquid dewP calculation for initial guess of P and liquid phase composition
    Tguess, xguess= dewT_ideal(y,P)

    #f (function to zero) maps values from -inf to inf, to values between 0 and 1
    #so xguess is mapped via logit which is the inverse function of expit
    v0 = np.r_[Tguess, logit(xguess)]
    res=root(f, v0)
    if not(res.success):
        return "FAILURE", res
    return res.x[0], expit(res.x[1:])

In [ ]:
def flash_idealPT(z, P, T):

    K=p.Pvap(T)/P
    def rachford(VF):
        return np.sum(z*(K-1)/(VF*(K-1) +1))

    res=root_scalar(rachford, bracket=(0,1))
    VF = res.root
    x=z/(1-VF + K *VF)
    y=K*x
    return x, y, VF

In [ ]:
def flash_NRTL_PT(z, P, T, maxiter = 100, tol=1e-12):

    dewP, dewx = dewP_NRTL(z, T)
    bubbleP, bubbley = bubbleP_NRTL(z,T)

    xguess = (P-dewP)/(bubbleP-dewP) * (z - dewx) +  dewx

    for i in range(maxiter):
        K=gamma(xguess,T)*p.Pvap(T)/P

        def rachford(VF):
            return np.sum(z*(K-1)/(VF*(K-1) +1))

        res=root_scalar(rachford, bracket=(0,1))
        VF = res.root
        x=z/(1-VF + K *VF)
        if (np.linalg.norm(xguess-x)<tol):
            break
        xguess = x

    y=K*x
    return x, y, VF, i

In [ ]:
def flash_NRTL_PT(z, P, T, maxiter = 100, tol=1e-12):

    dewP, dewx = dewP_NRTL(z, T)
    bubbleP, bubbley = bubbleP_NRTL(z,T)

    xguess = (P-dewP)/(bubbleP-dewP) * (z - dewx) +  dewx

    for i in range(maxiter):
        K=gamma(xguess,T)*p.Pvap(T)/P

        def rachford(VF):
            return np.sum(z*(K-1)/(VF*(K-1) +1))

        res=root_scalar(rachford, bracket=(0,1))
        VF = res.root
        x=z/(1-VF + K *VF)
        if (np.linalg.norm(xguess-x)<tol):
            break
        xguess = x

    y=K*x
    return x, y, VF, i



In [ ]:
Nc = p.Mw.size
Ftot = 1.
z = np.array([0.3,0.3,0.4])
P = 1e5
dewT, _ = dewT_NRTL(z, P)
bubbleT, _ = bubbleT_NRTL(z,P)
feedT = (bubbleT + dewT)/2
x,y, vf, _ = flash_NRTL_PT(z, P, feedT)
feedH=p.Hv(Ftot*vf*y, feedT) + p.Hl(Ftot*(1-vf)*x,feedT)

In [ ]:
D = 0.65*Ftot
Btot = Ftot - D
R = 10.
Ns = 40
Nf = 20

unk = np.zeros((Ns, 2*Nc+1))
Ltot_rec = R*D
Ltot_strip = Ltot_rec + Ftot*(1-vf)
Vtot_rec = R*D + D
Vtot_strip = Vtot_rec - Ftot*vf
unk[:Nf,:Nc] = Ltot_rec*y
unk[Nf:-1,:Nc] = Ltot_strip*x
unk[-1,:Nc] = Btot*x
unk[:Nf,Nc:2*Nc ] = Vtot_rec*y
unk[Nf:,Nc:2*Nc ] = Vtot_strip*x
unk[:,-1]=np.linspace(bubbleT_NRTL(y,P)[0],dewT_NRTL(x,P)[0],Ns)

In [ ]:
def stage1(vec, vec2, refluxT):
    T,T2 = vec[-1], vec2[-1]
    L,V = jnp.split(vec[:-1],2)
    L2,V2 = jnp.split(vec2[:-1],2)
    x = L/jnp.sum(L)
    y = V/jnp.sum(V)

    MB = (V2 + R*D*y - V - L)/Ftot
    EQ = x*p.NRTL_gamma(x,T)*p.Pvap(T)/P - y
    EB = (p.Hv(V2, T2) + p.Hl(R*D*y, refluxT) - p.Hv(V, T) - p.Hl(L, T))/feedH

    return jnp.r_[MB, EQ, EB]

def stage(vec1, vec, vec2, f, fH):
    T1, T,T2 = vec1[-1], vec[-1], vec2[-1]
    L1,V1 = jnp.split(vec1[:-1],2)
    L,V = jnp.split(vec[:-1],2)
    L2,V2 = jnp.split(vec2[:-1],2)
    x = L/jnp.sum(L)
    y = V/jnp.sum(V)

    MB = (f + V2 + L1 - V - L)/Ftot
    EQ = x*p.NRTL_gamma(x,T)*p.Pvap(T)/P - y
    EB = (fH + p.Hv(V2, T2) + p.Hl(L1, T1) - p.Hv(V, T) - p.Hl(L, T))/feedH

    return jnp.r_[MB, EQ, EB]


def stageN(vec1, vec):
    T1, T = vec1[-1], vec[-1]
    L1,V1 = jnp.split(vec1[:-1],2)
    L,V = jnp.split(vec[:-1],2)
    x = L/jnp.sum(L)
    y = V/jnp.sum(V)

    MB = (L1 - V - L)/Ftot
    EQ = x*p.NRTL_gamma(x,T)*p.Pvap(T)/P - y
    MB2 = (jnp.sum(L1) - jnp.sum(V) - Btot)/Ftot

    return jnp.r_[MB, EQ, MB2]

In [ ]:
stage1_jac = jax.jit(jax.jacobian(stage1, (0,1)))
stage_jac = jax.jit(jax.jacobian(stage, (0,1,2)))
stageN_jac = jax.jit(jax.jacobian(stageN, (0,1)))

In [ ]:
Es = np.zeros((Ns, 2*Nc+1))
Cs = np.zeros((Ns-1, 2*Nc+1, 2*Nc+1))

In [ ]:
def evalEs(unk):

    L,V = np.split(unk[0,:-1],2)
    y=V/np.sum(V)
    Es[0] = stage1(unk[0], unk[1], bubbleT_NRTL(y, P)[0])
    for i in range(1, Ns-1):
        if i==Nf:
            Es[i]= stage(unk[i-1], unk[i], unk[i+1], Ftot*z, feedH)
        else:
            Es[i]= stage(unk[i-1], unk[i], unk[i+1], 0., 0.)

    Es[-1] = stageN(unk[-2], unk[-1])

    return Es

def norm_evalEs(unk):
    return np.linalg.norm(evalEs(unk))


In [ ]:
for iter in range(1,25):
    Es= evalEs(unk)
    normEs = norm_evalEs(unk)
    print(iter, normEs)
    if normEs<1e-8:
        break
    B,C = stage1_jac(unk[0], unk[1],bubbleT_NRTL(y, P)[0])
    Binv = np.linalg.inv(B)
    Cs[0]= Binv @ C
    Es[0]= Binv @ Es[0]
    for i in range(1, Ns-1):
        if i==Nf:
            A,B,C = stage_jac(unk[i-1], unk[i], unk[i+1], Ftot*z, feedH)
        else:
            A,B,C= stage_jac(unk[i-1], unk[i], unk[i+1], 0., 0.)

        Binv = np.linalg.inv(B-A@Cs[i-1])
        Cs[i]=Binv@C
        Es[i]=Binv@(Es[i]-A@Es[i-1])

    A,B = stageN_jac(unk[-2], unk[-1])
    Binv = np.linalg.inv(B-A@Cs[-1])
    Es[-1] = Binv@ (Es[-1] - A@Es[-2])

    delta_unk = np.zeros_like(unk)
    delta_unk[-1]= Es[-1]
    for i in range(Ns-2,-1,-1):
        delta_unk[i]= Es[i]-Cs[i] @ delta_unk[i+1]

    brac = bracket(lambda t: norm_evalEs(unk + t*delta_unk), xa=0., xb=1.)
    t = minimize_scalar(lambda t: norm_evalEs(unk + t*delta_unk), brac[:3]).x
    unk = unk + t*delta_unk

1 1.9811943192565329
2 0.11243379405883172
3 0.015757070087462792
4 4.951382331597321e-05
5 8.512264481999099e-08
6 1.4782208004630679e-12


In [ ]:
L,V = np.split(unk[:,:-1],2, axis=1)
x=L/np.sum(L, axis=1)[:,None]
y=V/np.sum(V, axis=1)[:,None]

In [ ]:
x

array([[0.40412, 0.35043, 0.24545],
       [0.3856 , 0.36474, 0.24966],
       [0.36927, 0.37737, 0.25336],
       [0.35491, 0.38848, 0.25661],
       [0.34232, 0.39821, 0.25947],
       [0.3313 , 0.40672, 0.26198],
       [0.32166, 0.41412, 0.26422],
       [0.31324, 0.42054, 0.26621],
       [0.3059 , 0.42608, 0.26803],
       [0.29948, 0.43081, 0.26971],
       [0.29388, 0.4348 , 0.27132],
       [0.28897, 0.4381 , 0.27294],
       [0.28465, 0.44071, 0.27463],
       [0.28082, 0.44264, 0.27653],
       [0.27739, 0.44383, 0.27878],
       [0.27425, 0.44416, 0.2816 ],
       [0.27128, 0.44343, 0.28529],
       [0.26835, 0.44131, 0.29034],
       [0.26524, 0.43727, 0.29749],
       [0.26163, 0.43035, 0.30801],
       [0.25692, 0.41887, 0.32421],
       [0.24833, 0.42584, 0.32583],
       [0.23986, 0.43272, 0.32742],
       [0.23152, 0.43949, 0.32898],
       [0.22332, 0.44616, 0.33052],
       [0.21524, 0.45272, 0.33204],
       [0.20731, 0.45917, 0.33353],
       [0.19951, 0.46549, 0.

In [ ]:
y

array([[0.42506, 0.33429, 0.24065],
       [0.40603, 0.34896, 0.24501],
       [0.38919, 0.36197, 0.24884],
       [0.37435, 0.37344, 0.2522 ],
       [0.36131, 0.38354, 0.25515],
       [0.34987, 0.39238, 0.25775],
       [0.33985, 0.40011, 0.26004],
       [0.3311 , 0.40684, 0.26206],
       [0.32345, 0.41267, 0.26388],
       [0.31678, 0.41769, 0.26553],
       [0.31096, 0.42199, 0.26705],
       [0.30587, 0.42561, 0.26852],
       [0.30141, 0.42861, 0.26998],
       [0.29749, 0.43098, 0.27153],
       [0.29401, 0.43273, 0.27325],
       [0.2909 , 0.43381, 0.27529],
       [0.28804, 0.4341 , 0.27785],
       [0.28535, 0.43344, 0.28121],
       [0.28269, 0.43152, 0.28579],
       [0.27988, 0.42784, 0.29229],
       [0.27661, 0.42155, 0.30184],
       [0.26742, 0.42901, 0.30357],
       [0.25836, 0.43637, 0.30527],
       [0.24943, 0.44363, 0.30694],
       [0.24063, 0.45079, 0.30859],
       [0.23197, 0.45783, 0.3102 ],
       [0.22345, 0.46476, 0.31179],
       [0.21507, 0.47157, 0.

In [ ]:
opx = np.r_[y[0], np.repeat(x[:-1],2), x[-1]]
opx

array([0.41762, 0.33703, 0.24535, 0.39643, 0.39643, 0.352  , 0.352  ,
       0.25156, 0.25156, 0.37911, 0.37911, 0.36366, 0.36366, 0.25723,
       0.25723, 0.36491, 0.36491, 0.37245, 0.37245, 0.26264, 0.26264,
       0.35321, 0.35321, 0.37869, 0.37869, 0.26809, 0.26809, 0.34345,
       0.34345, 0.38261, 0.38261, 0.27394, 0.27394, 0.33512, 0.33512,
       0.38428, 0.38428, 0.2806 , 0.2806 , 0.32776, 0.32776, 0.38361,
       0.38361, 0.28863, 0.28863, 0.32086, 0.32086, 0.38028, 0.38028,
       0.29886, 0.29886, 0.3138 , 0.3138 , 0.3736 , 0.3736 , 0.3126 ,
       0.3126 , 0.30563, 0.30563, 0.36219, 0.36219, 0.33217, 0.33217,
       0.29455, 0.29455, 0.34318, 0.34318, 0.36226, 0.36226, 0.29064,
       0.29064, 0.34636, 0.34636, 0.36299, 0.36299, 0.28622, 0.28622,
       0.34988, 0.34988, 0.36389, 0.36389, 0.28121, 0.28121, 0.3537 ,
       0.3537 , 0.36509, 0.36509, 0.27545, 0.27545, 0.35767, 0.35767,
       0.36688, 0.36688, 0.26866, 0.26866, 0.36132, 0.36132, 0.37002,
       0.37002, 0.26

In [ ]:
opy=np.r_[np.repeat(y,2)]

In [ ]:
P=101325
x1s=np.linspace(0,1,101)
Ts=[]
y1s=[]
for x1 in x1s:
    T, (y1,y2) = bubbleT_NRTL(np.array([x1, 1-x1]), P)
    Ts.append(T)
    y1s.append(y1)

In [ ]:
fig = make_subplots()
fig.add_scatter(x=x1s, y=y1s)
fig.add_scatter(x=opx, y=opy)
fig.update_layout(width=600, height=600, template='plotly_dark')
